# DAM Description Extraction Worker 3/12
Deterministic distributed worker 3 of 12 for video-level object descriptions with atomic rclone sync.

In [ ]:
import os, subprocess
from pathlib import Path

target = Path('/kaggle/working/AIC-2026')
branch = 'feature/dam-text-extraction'
repo_url = 'https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git'

if not target.is_dir():
    subprocess.run(['git', 'clone', '-b', branch, repo_url, str(target)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin'], cwd=str(target), check=True)
    subprocess.run(['git', 'checkout', branch], cwd=str(target), check=True)
    subprocess.run(['git', 'pull', 'origin', branch], cwd=str(target), check=True)

os.chdir(target)
print('✅ Repo updated on branch:', branch)

In [ ]:
import os, re
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# 1. Install rclone binary
!apt-get update -qq && apt-get install -y -qq rclone

# 2. Write rclone config from Secret
try:
    raw_secret = UserSecretsClient().get_secret('RCLONE_CONFIG_GDRIVE').strip()
    if not raw_secret.startswith('['):
        raw_secret = '[gdrive] ' + raw_secret
    formatted = raw_secret
    if '\n' not in formatted:
        formatted = formatted.replace('[gdrive]', '[gdrive]\n')
        formatted = re.sub(r'\s+(type\s*=)', r'\n\1', formatted)
        formatted = re.sub(r'\s+(scope\s*=)', r'\n\1', formatted)
        formatted = re.sub(r'\s+(token\s*=)', r'\n\1', formatted)
        formatted = re.sub(r'\s+(client_id\s*=)', r'\n\1', formatted)
        formatted = re.sub(r'\s+(client_secret\s*=)', r'\n\1', formatted)
    for path_str in ['/root/.config/rclone/rclone.conf', '/root/.rclone.conf']:
        p = Path(path_str)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(formatted, encoding='utf-8')
    print('✓ rclone.conf written successfully!')
except Exception as exc:
    print(f'❌ Error: {exc}')

In [ ]:
!python -m pip install --quiet --no-deps -r requirements/kaggle.txt

In [ ]:
WORKER_ID = 3
NUM_WORKERS = 12
KEYFRAMES_ROOT = '/kaggle/input/datasets/lyduchoang/aic-26-video/Keyframes/Keyframes'
OBJECTS_ROOT = '/kaggle/input/datasets/khoalequangminh/aic-test-dataset/data/objects'
MAP_ROOT = '/kaggle/input/datasets/khoalequangminh/aic-test-dataset/data/map-keyframes'
RCLONE_DEST = 'gdrive:AIC_HCM/artifacts/dam_descriptions/'

# 1-second dry run verification
!python scripts/run_dam_batch.py \
  --worker-id {WORKER_ID} \
  --num-workers {NUM_WORKERS} \
  --keyframes-root {KEYFRAMES_ROOT} \
  --objects-root {OBJECTS_ROOT} \
  --map-keyframes-root {MAP_ROOT} \
  --rclone-dest {RCLONE_DEST} \
  --dry-run

In [ ]:
WORKER_ID = 3
NUM_WORKERS = 12
KEYFRAMES_ROOT = '/kaggle/input/datasets/lyduchoang/aic-26-video/Keyframes/Keyframes'
OBJECTS_ROOT = '/kaggle/input/datasets/khoalequangminh/aic-test-dataset/data/objects'
MAP_ROOT = '/kaggle/input/datasets/khoalequangminh/aic-test-dataset/data/map-keyframes'
RCLONE_DEST = 'gdrive:AIC_HCM/artifacts/dam_descriptions/'

# Launch full automated batch run
!python scripts/run_dam_batch.py \
  --worker-id {WORKER_ID} \
  --num-workers {NUM_WORKERS} \
  --keyframes-root {KEYFRAMES_ROOT} \
  --objects-root {OBJECTS_ROOT} \
  --map-keyframes-root {MAP_ROOT} \
  --rclone-dest {RCLONE_DEST} \
  --output-root /kaggle/working/aic2026-artifacts \
  --cache-root /kaggle/working/aic2026-model-cache \
  --device cuda